In [ ]:
"""
sandbox_time.ipynb

A sandbox to develop a time-resolved class.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

## init

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
)

encoder_mb = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
    strategy_filter="mf",
)

In [ ]:
encoder.verify()
encoder_mb.verify()
encoder_mf.verify()

In [ ]:
encoder.view_fits(model="baseline")

In [ ]:
encoder.view_fits()

In [ ]:
encoder.view_weights()

## trajectories

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
)

encoder_mb = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
    strategy_filter="mf",
)

encoder.fit_encoder()
encoder_mb.fit_encoder()
encoder_mf.fit_encoder()

In [ ]:
from sg.models import Bootstrapper

bs_mb = Bootstrapper(
    subj_id,
    sess_id,
    make_tre(StrategyEncoder, tr_type="dme"),
    n=100,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mb",
)
bs_mf = Bootstrapper(
    subj_id,
    sess_id,
    make_tre(StrategyEncoder, tr_type="dme"),
    n=100,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mf",
)

bs_mb.get_bweight_stats()
bs_mf.get_bweight_stats()

In [ ]:
np.all(
    [
        np.all(encoder.spike_times[reg][i] == encoder_mf.spike_times[reg][i])
        for reg in ["DMS", "DLS"]
        for i in range(encoder.psths[reg].shape[0])
    ]
)

In [ ]:
trial_data = encoder.trial_data

lc_mask = (trial_data.response == 1) & (trial_data.rewarded)
rc_mask = (trial_data.response == -1) & (trial_data.rewarded)
li_mask = (trial_data.response == 1) & (~trial_data.rewarded)
ri_mask = (trial_data.response == -1) & (~trial_data.rewarded)

assert np.isclose(lc_mask.mean() + rc_mask.mean() + li_mask.mean() + ri_mask.mean(), 1)

c_mask = trial_data.rewarded == 1
i_mask = trial_data.rewarded == 0

assert np.all(c_mask == (lc_mask) | (rc_mask)) and np.all(
    i_mask == (li_mask) | (ri_mask)
)

In [ ]:
psth_i = encoder.psths["DLS"][7, i_mask].mean(axis=0)
psth_c = encoder.psths["DLS"][7, c_mask].mean(axis=0)

plt.figure()
plt.plot(psth_c)
plt.plot(psth_i)
plt.show()

In [ ]:
plt.figure()
plt.imshow(encoder.psths["DLS"][7, c_mask], aspect="auto")
plt.show()

In [ ]:
encoder.view_peths(mode="rewarded", reg="DLS")

In [ ]:
encoder_mf.view_peths()

In [ ]:
from squiggs.renderers import StrategyWeightPETHRenderer
from squiggs.neuron_viewer import NeuronViewer
from core.data import get_psths_cond, tv_pos_neg

mode = "rewardef"
regressor = mode
reg = "DLS"
# rewd 33, (dms), 1, 60, 76, (dls)
# resp 6, 30, 36, 48, 49 (dms), 1, 4, 5, 63(dls)

r = StrategyWeightPETHRenderer(
    bootstrapper_mb=bs_mb,
    bootstrapper_mf=bs_mf,
    encoder_mb=encoder_mb,
    encoder_mf=encoder_mf,
    reg=reg,
    regressor=regressor,
    values=(tv_pos_neg[regressor]["pos"], tv_pos_neg[regressor]["neg"]),
    peths_mb=get_psths_cond(encoder_mb.psths[reg], encoder_mb.trial_data, mode=mode),
    peths_mf=get_psths_cond(encoder_mf.psths[reg], encoder_mf.trial_data, mode=mode),
    pres=encoder.tpre,
    posts=encoder.tpost,
    binwidth_s=encoder.stepsize_s,
    tbin_centers=encoder.tbin_centers,
)

# visualize scatter as function of timepoint
# plot other regressor trajectories on the same plot
# representational geometry in mb vs mf trials (parallel vs orthogonal)
# -> is population reprenstation different across strategies

nv = NeuronViewer(num_units=encoder.psths[reg].shape[0], render_func=r)

## response only

In [ ]:
from sg.models import make_tre, Encoder

encoder_response = make_tre(Encoder)(
    subj_id,
    sess_id,
    tv_keys=["response"],
    norm=False,
    stepsize_s=0.1,
)

In [ ]:
encoder_response.view_weights()